# Segment analysis

Where the model wins and where it breaks. AUC and calibration error are reported per segment; any segment whose calibration drifts is flagged, because the pricing layer trusts these probabilities segment by segment.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.data import INTERIM, ROOT
from src.models import time_split

FIGURES = ROOT / "figures"; FIGURES.mkdir(exist_ok=True)
CUTOFF, CATS = "2015-07-01", ["deviceID", "paymentMethod"]
DROP = ["orderID", "orderDate", "customerID", "any_return"]

def gbm():
    return lgb.LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                              n_jobs=-1, verbose=-1)

df = pd.read_parquet(INTERIM / "orders.parquet")
for block in ["basket", "customer", "article"]:
    df = df.merge(pd.read_parquet(INTERIM / f"{block}.parquet"), on="orderID", how="left")
FEATS = [c for c in df.columns if c not in DROP]
X = df[FEATS].copy()
for c in CATS:
    X[c] = X[c].astype("category")
y = df.any_return
tr, va = time_split(df, "orderDate", CUTOFF)
df.shape, len(FEATS)

((738698, 40), 36)

## Score

**Refit and score.** The notebook is self-contained: it rebuilds the same fixed model rather than depending on a pickle from notebook 07, so it can be run in isolation.

In [2]:
from sklearn.metrics import roc_auc_score
m = gbm().fit(X[tr], y[tr])
pred = m.predict_proba(X[va])[:, 1]
val = df.loc[va, ["orderID", "n_lines", "paymentMethod", "any_return"]].copy()
val["pred"] = pred
val["cust_prior_orders"] = df.loc[va, "cust_prior_orders"].values
val.shape

(113966, 6)

## Segments

**Attach segment attributes.** Product group is taken as the first line's group per order — a simplification, labelled as such, since a basket can span categories.

In [3]:
from src.data import load_orders
groups = (load_orders("orders_train").groupby("orderID").productGroup.first().rename("product_group"))
val = val.merge(groups, on="orderID", how="left")

top_groups = val.product_group.value_counts().head(5).index.tolist()
val["basket"] = np.where(val.n_lines > 1, "multi-line", "single-line")
val["customer"] = np.where(val.cust_prior_orders > 0, "returning", "new")
val["group_seg"] = np.where(val.product_group.isin(top_groups), "group " + val.product_group.astype(str), "other")
top_pay = val.paymentMethod.value_counts().head(5).index.tolist()
val["pay_seg"] = np.where(val.paymentMethod.isin(top_pay), val.paymentMethod, "other")
top_groups

[3.0, 8.0, 2.0, 1.0, 13.0]

**Per-segment AUC and calibration.** The guard returns NaN for segments too small or too one-sided for ten bins. Segments above 0.02 ECE are flagged `CHECK`: a model can rank well inside a segment while being systematically mis-levelled there, and the pricing layer would inherit that error silently.

In [4]:
def ece(d, bins=10):
    if d.pred.nunique() < bins or d.any_return.nunique() < 2:
        return np.nan
    b = pd.qcut(d.pred, bins, labels=False, duplicates="drop")
    g = d.groupby(b).agg(n=("pred", "size"), p=("pred", "mean"), a=("any_return", "mean"))
    return float((g.n / g.n.sum() * (g.p - g.a).abs()).sum())

def summarise(col):
    out = []
    for name, d in val.groupby(col):
        out.append({"dimension": col, "segment": str(name), "n": len(d),
                    "base_rate": d.any_return.mean(),
                    "auc": roc_auc_score(d.any_return, d.pred) if d.any_return.nunique() > 1 else np.nan,
                    "ece": ece(d)})
    return pd.DataFrame(out)

seg = pd.concat([summarise(c) for c in ["basket", "customer", "group_seg", "pay_seg"]], ignore_index=True)
seg["calibration_flag"] = np.where(seg.ece > 0.02, "CHECK", "")
seg.sort_values(["dimension", "n"], ascending=[True, False]).round(4)

,dimension,segment,n,base_rate,auc,ece,calibration_flag
0,basket,multi-line,84771,0.7260,0.8584,0.0062,
1,basket,single-line,29195,0.3529,0.6960,0.0086,
3,customer,returning,79030,0.6525,0.8621,0.0050,
2,customer,new,34936,0.5803,0.8265,0.0056,
7,group_seg,group 3.0,46766,0.6435,0.8640,0.0109,
9,group_seg,other,24982,0.6395,0.8477,0.0086,
8,group_seg,group 8.0,18676,0.6383,0.8584,0.0126,
6,group_seg,group 2.0,8140,0.5746,0.8412,0.0157,
4,group_seg,group 1.0,7900,0.5322,0.8097,0.0146,
5,group_seg,group 13.0,7502,0.6624,0.8116,0.0106,


## Save

**Persist and surface the failures.** The final view filters to segments flagged `CHECK`, so the notebook ends on its own weak spots rather than its averages. Both flagged segments are small payment methods where calibration drifts past 0.04 — the model still ranks well there, but the probability *levels* are off, and the pricing layer would inherit that error silently.

In [5]:
seg.to_csv(INTERIM / "segments.csv", index=False)
seg.loc[seg.calibration_flag == "CHECK", ["dimension", "segment", "n", "base_rate", "auc", "ece"]].round(4)

,dimension,segment,n,base_rate,auc,ece
10,pay_seg,BPLS,1841,0.4199,0.8408,0.0427
15,pay_seg,other,2623,0.4602,0.7808,0.0406
